In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
import os
load_dotenv(override=True)  # override=True 确保 .env 会覆盖系统已有的同名变量
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))  # 确认代理是否生效
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

In [ ]:
# 基础用法
@tool
def search_weather(query:str,limit:int=10)->str:
    """Search for weather information based on a query.
    
    Args:
        query (str): The query to search for.
        limit (int): The maximum number of results to return.
    """
    return f"found {limit} results for {query}"

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

In [2]:
load_dotenv(override=True)  # override=True 确保 .env 会覆盖系统已有的同名变量
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))  # 确认代理是否生效
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

HTTPS_PROXY: http://127.0.0.1:7897


In [3]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [5]:
@tool("web_seach")
def search(query:str)-> str:
    """Search the web for information"""
    return f"Result for {query}"
print(search.name)

web_seach


In [12]:
@tool("calculator",description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression:str)->str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))
print(calculator.description)
calculator.invoke("2+3")               # → "5"


Performs arithmetic calculations. Use this for any math problems.


'5'

In [16]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [ ]:
# runtime 不消耗token从内存中读取信息
from langchain.tools import tool,ToolRuntime
from langchain.messages import HumanMessage
@tool
def get_last_user_message(runtime:ToolRuntime)->str:
    """Get the most message from the user."""
    messages = runtime.state["messages"]
    for message in reversed(messages):
        if isinstance(message,HumanMessage):
            return message.content
    return "No user messages found"
@tool
def get_user_preference(
    pref_name:str,
    runtime:ToolRuntime,
)->str:
    """Get a user preference by name."""
    preferences = runtime.state["preferences"]
    return preferences.get(pref_name,"No preference found")

In [21]:
from langchain.agents import AgentState
from langchain.messages import ToolMessage
from langchain.tools import tool,ToolRuntime
from langgraph.types import Command

class CustomState(AgentState):
    user_name:str

@tool
def set_user_name(new_name:str,runtime:ToolRuntime[None,CustomState])->str:
    """Set the user's name."""
    return Command(
        update={
            "user_name":new_name,
            "message":[
                ToolMessage(
                    content=f"User name set to {new_name}",
                    tool_call_id = runtime.tool_call_id
                )
            ]
        }
    )


In [27]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool,ToolRuntime
from langchain_core.utils.uuid import uuid7
from langchain_openrouter import ChatOpenRouter

USER_DATABASE = {
    "DTEST001": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com",
    },
}

@dataclass
class UserContext:
    user_id:str

@tool
# 好好笑 这里偷偷传递
def get_account_info(runtime:ToolRuntime[UserContext]) ->str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id
    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Account holder:{user['name']}"
            f"Type:{user['account_type']}"
            f"Blance:{user['balance']}"
        )
    return "user not found "

model = ChatOpenRouter(model="deepseek/deepseek-v4-flash-0731")
agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt = "You are a financial assisant."
)
result = agent.invoke(
    {"messages":[{"role":"user","content":"what's my current balance"}]},
    config={"configurable":{"thread_id":str(uuid7())}},
    context = UserContext(user_id="DTEST001")
)
print(result)


{'messages': [HumanMessage(content="what's my current balance", additional_kwargs={}, response_metadata={}, id='eaee844e-24d8-4b1f-85c0-3b30668aff25'), AIMessage(content="I'll check your account information to find", additional_kwargs={'reasoning_content': 'The user wants to know their current balance. I need to fetch account info. Let me call get_account_info.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants to know their current balance. I need to fetch account info. Let me call get_account_info.'}]}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-0731', 'id': 'gen-1787207868-UAnRSkj8rj6YDAFwubXS', 'created': 1787207868, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.3874e-05, 'cost_details': {'upstream_inference_completions_cost': 8.96e-06, 'upstream_inference_prompt_cost': 4.914e-06, 'upstream_inference_cost': 1.3874e-05}}, id='lc_run--01

In [29]:
# 使用suer_info进行存储
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool,ToolRuntime
from langchain_openrouter import ChatOpenRouter

@tool
def get_user_info(user_id:str,runtime:ToolRuntime)->str:
    """Look up user info"""
    store = runtime.store
    user_info = store.get(("users",),user_id)
    return str(user_info.value) if user_info else "Unknown user"

@tool
def save_user_info(user_id:str,user_info:dict[str,Any],runtime:ToolRuntime) ->str:
    """save user info"""
    store = runtime.store
    store.put(("users",),user_id,user_info)
    return "Successfully saved user info"

model = ChatOpenRouter(model="deepseek/deepseek-v4-flash-0731")
store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info],
    store = store
)
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})


{'messages': [HumanMessage(content="Get user info for user with id 'abc123'", additional_kwargs={}, response_metadata={}, id='38195786-2891-47f1-af91-55d1c727452f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "I need to get user info for user 'abc123'. Let me call the get_user_info function.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "I need to get user info for user 'abc123'. Let me call the get_user_info function."}]}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-0731', 'id': 'gen-1787208887-2IpGbjpWPFH4dhhgYgKg', 'created': 1787208887, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 2.989e-05, 'cost_details': {'upstream_inference_completions_cost': 9.66e-06, 'upstream_inference_prompt_cost': 2.023e-05, 'upstream_inference_cost': 2.989e-05}}, id='lc_run--01a01df3-6dec-73d0-8ed2-c76f17b55f05-0', tool_calls=[{'name': 'get_user_info'

In [30]:
from langchain.tools import tool,ToolRuntime
@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer

    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")

    return f"It's always sunny in {city}!"

In [31]:
from langchain.tools import tool, ToolRuntime

@tool
def log_execution_context(runtime: ToolRuntime) -> str:
    """Log execution identity information."""
    info = runtime.execution_info
    print(f"Thread: {info.thread_id}, Run: {info.run_id}")
    print(f"Attempt: {info.node_attempt}")
    return "done"

In [32]:
from langchain.tools import tool,ToolRuntime
@tool
def get_assistant_scoped_data(runtime:ToolRuntime) -> str:
    """fetch data scoped to the current assistant"""
    server = runtime.server_info
    if server is not None:
        print(f"Assistant: {server.assistant_id}, Graph: {server.graph_id}")
        if server.user is not None:
            print(f"User:{server.user.identity}")
    return "done"


In [ ]:
from langchain.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"It is currently sunny in {city}."

In [33]:
from collections.abc import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
from langchain.tools.tool_node import ToolCallRequest


@wrap_tool_call
def handle_tool_errors(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage],
) -> ToolMessage:
    """Convert tool exceptions into ToolMessages the model can handle."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({e})",
            tool_call_id=request.tool_call["id"],
        )


agent = create_agent(
    model="openrouter:z-ai/glm-5.2",
    tools=[],
    middleware=[handle_tool_errors],
)